# 1-Million Record Memory Explosion: Autopsy and Streaming Solutions

This notebook analyses memory consumption when loading large datasets,
comparing loading all records into a list with streaming records
incrementally through generators.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/04_One_Million_Record_Streaming/` project folder (anchored on `generate_dataset.py` so its modules import cleanly) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "04_One_Million_Record_Streaming"
ANCHOR_REL = "generate_dataset.py"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Generate the Sample Dataset
The CSV is generated inside the project folder (git-ignored); 100k rows are
used for a fast interactive demo while the same code scales to 1M.


In [ ]:
import os
import tracemalloc

from generate_dataset import generate_large_csv
from streaming_solutions import read_all_inefficient, stream_records

filename = str(PROJ_DIR / "large_patients.csv")
generate_large_csv(filename, total_rows=100000)
print(f"File size on disk: {os.path.getsize(filename) / (1024 * 1024):.2f} MB")


### 2. Inefficient List Allocation

Calling `list(reader)` creates hundreds of thousands of dictionary objects in memory at once.


In [ ]:
tracemalloc.start()
bad_data = read_all_inefficient(filename)
_, peak_bad = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak Memory Used: {peak_bad / (1024 * 1024):.2f} MB")
del bad_data


### 3. Generator Streaming Solution

Yielding records one at a time keeps memory usage flat at a few kilobytes, regardless of file size.


In [ ]:
tracemalloc.start()
count_high_bp = 0
for row in stream_records(filename):
    if int(row["BloodPressure"]) > 140:
        count_high_bp += 1

_, peak_good = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak Memory Used: {peak_good / (1024 * 1024):.4f} MB")
print(f"Hypertensive patients counted: {count_high_bp:,}")
